# 트랜스포머의 그래디언트 흐름
## 잔차 연결과 그래디언트 고속도로

| 항목 | 내용 |
|------|------|
| Tutorial ID | `adv-3-1` / Section `adv-3-1-1` |
| 대상 | 딥러닝 역전파를 처음 깊이 공부하는 학습자 |
| 핵심 키워드 | 그래디언트 소실, 잔차 연결, 야코비안, 클리핑, 워밍업 |

## 이 노트북을 읽는 법

### 배경 지식 (읽기 전 알면 좋은 것)
- **편미분**: 여러 변수 중 하나를 기준으로 미분
- **연쇄 법칙**: 합성 함수를 미분할 때 각 함수의 미분을 곱함
- **행렬 곱**: `A @ B` 형태의 연산

### 학습 목표 (이 노트북을 끝내면 알 수 있는 것)
1. 깊은 신경망에서 **그래디언트가 왜 소실**되는지 수치로 확인
2. **잔차 연결(Skip Connection)** 이 그 문제를 어떻게 막는지 이해
3. **소프트맥스 야코비안**을 통해 어텐션 그래디언트의 특성 파악
4. **그래디언트 클리핑**으로 폭발을 어떻게 억제하는지 확인
5. **학습률 워밍업**이 초기 학습 불안정을 어떻게 완화하는지 이해

### 구성
```
섹션 0  │ 라이브러리 및 공통 함수
섹션 1  │ 그래디언트 소실 문제      ← 문제 정의
섹션 2  │ 잔차 연결                  ← 핵심 해결책
섹션 3  │ 소프트맥스의 그래디언트    ← 어텐션 레이어 심화
섹션 4  │ 그래디언트 클리핑          ← 폭발 방지
섹션 5  │ 학습률 워밍업              ← 학습 안정화
종합    │ 모든 기법 비교 실험
```

> **팁**: 각 셀의 파라미터(레이어 수, 온도, max_norm 등)를 바꿔보면서
> 결과가 어떻게 달라지는지 직접 실험해보세요!

In [ ]:
# ============================================================
# 섹션 0: 라이브러리 불러오기 및 공통 설정
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── 그래프 스타일 설정 ─────────────────────────────────────
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
plt.rcParams['grid.alpha'] = 0.35
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ── 재현성을 위한 시드 고정 ────────────────────────────────
# numpy 난수 생성기의 시작점을 고정합니다.
# 같은 seed = 항상 동일한 난수 시퀀스 = 결과 재현 가능
SEED = 42
np.random.seed(SEED)

print("=" * 55)
print("  라이브러리 불러오기 완료")
print("=" * 55)
print()
print("  이 노트북에서 다룰 핵심 질문들:")
print("  Q1. 레이어가 깊어지면 그래디언트는 어떻게 될까?")
print("  Q2. 잔차 연결이 없으면 어떤 일이 생길까?")
print("  Q3. 어텐션이 집중될수록 역전파에 어떤 영향이?")
print("  Q4. 그래디언트가 폭발할 때 어떻게 막을까?")
print("  Q5. 학습 초기에 왜 lr을 작게 시작할까?")

## 섹션 1: 그래디언트 소실 문제

### 역전파의 핵심: 연쇄 법칙 (Chain Rule)

딥러닝 학습은 **역전파(Backpropagation)** 를 통해 이루어집니다.

```
[순전파]  입력 → 레이어₁ → 레이어₂ → ... → 레이어ₙ → 손실(Loss)
[역전파]  손실 ← 레이어ₙ ← ... ← 레이어₂ ← 레이어₁ ← 그래디언트
```

역전파는 **연쇄 법칙**으로 각 레이어의 그래디언트를 계산합니다.

예: 3개 레이어의 경우
```
dL/dW₁ = dL/dy₃ × dy₃/dy₂ × dy₂/dy₁ × dy₁/dW₁
          └───────────────────────────────────────┘
             레이어마다 미분값을 계속 곱해야 함!
```

### 문제: 미분값을 계속 곱하면...

**Sigmoid** 활성화 함수를 예로 들면:
```
Sigmoid 미분의 최댓값 = 0.25   (x = 0 일 때)
```

| 레이어 수 | 그래디언트 크기 |
|-----------|-----------------|
| 10개      | 0.25¹⁰ ≈ 10⁻⁶  |
| 20개      | 0.25²⁰ ≈ 10⁻¹²  |
| 50개      | 0.25⁵⁰ ≈ 10⁻³⁰  |

→ 초기 레이어까지 그래디언트가 **거의 0**이 되어버립니다!  
→ 초기 레이어 가중치는 사실상 **학습이 되지 않습니다**.

이것이 바로 **그래디언트 소실(Vanishing Gradient)** 문제입니다.

In [ ]:
# ============================================================
# 섹션 1: 그래디언트 소실 - Sigmoid 분석
#
# sigmoid 함수는 0~1 사이 값을 출력하는 활성화 함수입니다.
# 미분값(기울기)이 최대 0.25이기 때문에,
# 레이어를 거칠 때마다 그래디언트가 최대 1/4로 줄어듭니다.
# ============================================================

print("=" * 55)
print("  섹션 1: 그래디언트 소실 (Vanishing Gradient)")
print("=" * 55)

# ── Sigmoid 함수와 미분 정의 ──────────────────────────────

def sigmoid(x):
    # 수치 안정성을 위해 클리핑 처리
    # exp(-x)가 너무 커지면 오버플로우가 발생하므로 입력 범위를 제한
    x_clipped = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x_clipped))

def sigmoid_grad(x):
    # Sigmoid 미분 공식: σ(x) × (1 - σ(x))
    # 이 값은 x=0일 때 최대(0.25), x가 크거나 작을수록 0에 가까워짐
    s = sigmoid(x)
    return s * (1.0 - s)

# ── Sigmoid 미분값 확인 ──────────────────────────────────

print()
print("  [Sigmoid 미분 특성 확인]")
print("  " + "-" * 45)
print(f"  {'x 값':>8} │ {'sigmoid(x)':>12} │ {'미분값':>10} │ 메모")
print("  " + "-" * 55)

for x_val in [0, 1, 2, 3, 5]:
    s_val = sigmoid(x_val)
    g_val = sigmoid_grad(x_val)
    note = "<-- 여기서 최대 (0.25)" if x_val == 0 else ("점점 작아짐..." if x_val > 0 else "")
    print(f"  {x_val:>8} │ {s_val:>12.4f} │ {g_val:>10.4f} │ {note}")

print()
print("  핵심: Sigmoid 미분은 항상 0.25 이하입니다!")
print("        레이어마다 그래디언트에 0.25 이하 값을 곱합니다.")

# ── 레이어 수에 따른 그래디언트 소실 시뮬레이션 ──────────

print()
print("  [레이어 수에 따른 그래디언트 크기]")
print("  초기 그래디언트 = 1.0 (역전파 시작)")
print("  각 레이어에서 sigmoid 최대 미분값(0.25) 적용")
print()
print(f"  {'레이어':>8} │ {'그래디언트':>14} │ 시각화 (로그 스케일)")
print("  " + "-" * 65)

SIGMOID_MAX_GRAD = 0.25  # sigmoid 미분의 최댓값

test_depths = [1, 5, 10, 20, 50, 100]
vanishing_data = {}  # 나중에 시각화에 사용

for n in test_depths:
    g = SIGMOID_MAX_GRAD ** n
    vanishing_data[n] = g

    # ASCII 바 차트 (로그 스케일)
    log_val = min(30, max(0, int(-np.log10(g + 1e-100) * 0.7)))
    bar = "█" * (30 - log_val) + "░" * log_val

    if g < 1e-10:
        status = "⚠️  학습 불가!"
    elif g < 1e-5:
        status = "⚡ 매우 위험"
    elif g < 0.01:
        status = "△ 위험 수준"
    else:
        status = "✓  양호"

    print(f"  {n:>8}개 │ {g:>14.3e} │ {bar} {status}")

print()
print("  결론: 50개 레이어면 그래디언트가 10^-30 수준으로 소실!")
print("        초기 레이어 가중치는 사실상 업데이트되지 않습니다.")

In [ ]:
# ── 시각화: 그래디언트 소실 ──────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("섹션 1: 그래디언트 소실 문제", fontsize=13, fontweight="bold", y=1.01)

# ── 왼쪽: 레이어 수 vs 그래디언트 크기 ──────────────────

ax1 = axes[0]
layers_x = np.arange(1, 101)

# Sigmoid 소실 곡선
sigmoid_curve = SIGMOID_MAX_GRAD ** layers_x
ax1.semilogy(layers_x, sigmoid_curve, color="#E74C3C", linewidth=2.5,
             label="Sigmoid 활성화 (소실)")

# 이상적인 경우 (그래디언트 = 1 유지)
ax1.semilogy(layers_x, np.ones_like(layers_x), color="#2980B9",
             linestyle="--", linewidth=2, label="이상적 그래디언트")

# 학습 불가 기준선
ax1.axhline(y=1e-10, color="#E67E22", linestyle=":", linewidth=1.8,
            label="학습 불가 기준 (10⁻¹⁰)")

# 소실 영역 강조
ax1.fill_between(layers_x, 1e-35, 1e-10, alpha=0.12, color="#E74C3C")
ax1.text(65, 1e-18, "그래디언트\n소실 영역", color="#E74C3C",
         fontsize=9, ha="center", style="italic")

# 특정 레이어 위치 표시
for n_mark, c_mark in [(20, "#9B59B6"), (50, "#C0392B")]:
    g_mark = SIGMOID_MAX_GRAD ** n_mark
    ax1.annotate(f"{n_mark}레이어\n→ {g_mark:.0e}",
                 xy=(n_mark, g_mark), xytext=(n_mark + 18, g_mark * 1000),
                 arrowprops=dict(arrowstyle="->", color=c_mark, lw=1.5),
                 fontsize=9, color=c_mark,
                 bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

ax1.set_xlabel("레이어 수", fontsize=11)
ax1.set_ylabel("그래디언트 크기 (로그 스케일)", fontsize=11)
ax1.set_title("레이어가 깊어질수록 그래디언트 소실")
ax1.legend(fontsize=9, loc="upper right")
ax1.set_xlim(1, 100)

# ── 오른쪽: Sigmoid 함수와 그 미분 ────────────────────

ax2 = axes[1]
x_range = np.linspace(-6, 6, 300)

ax2.plot(x_range, sigmoid(x_range), color="#2980B9", linewidth=2.5,
         label="Sigmoid  f(x)")
ax2.plot(x_range, sigmoid_grad(x_range), color="#E74C3C", linewidth=2.5,
         label="미분  f'(x)")
ax2.axhline(y=0.25, color="#E67E22", linestyle="--", linewidth=2,
            label="최댓값 = 0.25")

# 최댓값 표시
ax2.fill_between(x_range, 0, sigmoid_grad(x_range), alpha=0.12, color="#E74C3C")
ax2.annotate("최대 0.25\n(x = 0)",
             xy=(0, 0.25), xytext=(2.2, 0.22),
             arrowprops=dict(arrowstyle="->", color="#E67E22", lw=1.5),
             fontsize=10, color="#E67E22",
             bbox=dict(boxstyle="round", facecolor="#FEF9E7", alpha=0.9))

ax2.set_xlabel("입력값 x", fontsize=11)
ax2.set_ylabel("함숫값", fontsize=11)
ax2.set_title("Sigmoid와 그 미분:\n미분 최댓값 = 0.25 (항상!)")
ax2.legend(fontsize=9)
ax2.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig("sec1_vanishing_gradient.png", dpi=150, bbox_inches="tight")
plt.show()
print("그래프 저장: sec1_vanishing_gradient.png")

## 섹션 2: 잔차 연결 (Residual Connection)

### 핵심 아이디어: 지름길을 만들자

| 구조 | 수식 | 그래디언트 |
|------|------|-----------|
| 일반 레이어 | 출력 = F(x) | ∂F/∂x |
| **잔차 레이어** | **출력 = F(x) + x** | **∂F/∂x + 1** |

→ 잔차 연결은 그래디언트에 항상 **+1** 을 추가합니다!  
→ F의 그래디언트가 아무리 작아져도, **1이 그대로 통과**합니다.

### 비유: 그래디언트 고속도로

```
일반 방식:
  [입력] → [복잡한 레이어 F] → [출력]
               ↑ 역전파 시 이 길만 있음 (막힐 수 있음)

잔차 연결:
  [입력] → [복잡한 레이어 F] → [+] → [출력]
    └─────────────────────────┘
              고속도로!
              ↑ 역전파 시 이 길을 통해 그래디언트가 막힘없이 통과
```

### 트랜스포머에서의 실제 코드

```python
# Pre-LayerNorm 방식 (GPT 계열)
x = x + self.attention(self.ln1(x))   # ← 잔차 연결
x = x + self.ffn(self.ln2(x))         # ← 잔차 연결
```

→ 각 레이어마다 `x +` 가 붙어있는 것이 바로 잔차 연결입니다!

### 수학적으로 왜 효과적인가?

`출력 = F(x) + x` 를 x로 미분하면:

```
∂출력/∂x = ∂F/∂x + 1
```

50개 레이어를 거치면:
```
grad = Π (∂Fᵢ/∂x + 1)    (i=1..50)
     ≈ Π (작은 값 + 1)
     ≈ 1 × 1 × ... × 1 ≈ 1  (소실 안 함!)
```

In [ ]:
# ============================================================
# 섹션 2: 잔차 연결의 효과 실험
#
# 핵심 수식:
#   일반:  grad = grad × ∂F/∂x         (0.25 이하의 값을 계속 곱함)
#   잔차:  grad = grad × (∂F/∂x + 1)   (+1 덕분에 최소 1 보장)
# ============================================================

print("=" * 55)
print("  섹션 2: 잔차 연결 효과 비교 실험")
print("=" * 55)

# ── 실험 설정 ────────────────────────────────────────────

N_LAYERS = 50         # 레이어 수 (트랜스포머와 유사한 깊이)
N_EXPERIMENTS = 3000  # 반복 실험 횟수 (통계적 신뢰성)
DF_STD = 0.1          # F(x)의 그래디언트 표준편차

print()
print(f"  실험 설정:")
print(f"    레이어 수    : {N_LAYERS}개")
print(f"    실험 횟수    : {N_EXPERIMENTS}회")
print(f"    ∂F/∂x 분포  : 평균=0, 표준편차={DF_STD} (정규분포)")
print(f"    → F(x)의 그래디언트가 작고 불규칙한 상황을 가정")

# ── 방법 1: 잔차 없음 (Sigmoid 활성화) ───────────────────

print()
print("  [방법 1] 잔차 없음 - Sigmoid 활성화")
print("  " + "-" * 45)

grad_no_res = 1.0
for layer_i in range(N_LAYERS):
    # 각 레이어마다 sigmoid 최대 미분값(0.25)을 곱함
    # 실제론 입력에 따라 달라지지만, 여기선 보수적(최악) 추정
    grad_no_res *= SIGMOID_MAX_GRAD

print(f"  {N_LAYERS}개 레이어 통과 후 그래디언트: {grad_no_res:.2e}")
print(f"  → 완전히 소실! (거의 0)")

# ── 방법 2: 잔차 있음 (1 + ∂F/∂x) ───────────────────────

print()
print("  [방법 2] 잔차 있음 - (1 + ∂F/∂x)")
print("  " + "-" * 45)

np.random.seed(SEED)
residual_grads = []

for _ in range(N_EXPERIMENTS):
    grad = 1.0  # 역전파 시작점 그래디언트

    for _ in range(N_LAYERS):
        # 각 레이어에서 F(x)의 그래디언트: 작은 랜덤값
        dF_dx = DF_STD * np.random.randn()

        # 잔차 연결의 핵심:
        # 그래디언트 = ∂F/∂x + 1
        # ∂F/∂x가 0에 가깝더라도, +1 덕분에 최소 1은 전달됨
        grad *= (1.0 + dF_dx)

    residual_grads.append(grad)

residual_grads = np.array(residual_grads)

print(f"  {N_EXPERIMENTS}회 실험 결과:")
print(f"    평균      : {np.mean(residual_grads):.6f}")
print(f"    중앙값    : {np.median(residual_grads):.6f}")
print(f"    표준편차  : {np.std(residual_grads):.6f}")
print(f"    10~90 %   : [{np.percentile(residual_grads,10):.4f}, "
      f"{np.percentile(residual_grads,90):.4f}]")

print()
print("  " + "=" * 45)
print("  비교 결과:")
print(f"    잔차 없음: {grad_no_res:.2e}  (소실)")
print(f"    잔차 있음: {np.mean(residual_grads):.6f}  (보존!)")
ratio = abs(np.mean(residual_grads)) / max(abs(grad_no_res), 1e-40)
print(f"    개선 배수: 약 {ratio:.0e}배")
print("  " + "=" * 45)

In [ ]:
# ── 레이어별 그래디언트 추이 계산 ───────────────────────

print("레이어별 평균 그래디언트 계산 중... (잠시 기다려주세요)")

np.random.seed(SEED)
layer_range = np.arange(1, 101)
mean_residual_by_depth = []

for depth in layer_range:
    samples = []
    for _ in range(500):
        g = 1.0
        for _ in range(depth):
            g *= (1.0 + DF_STD * np.random.randn())
        samples.append(g)
    mean_residual_by_depth.append(np.mean(samples))

mean_residual_by_depth = np.array(mean_residual_by_depth)
print("계산 완료!")

# ── 시각화 ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("섹션 2: 잔차 연결의 효과", fontsize=13, fontweight="bold", y=1.01)

# 왼쪽: 레이어 수 vs 평균 그래디언트
ax1 = axes[0]
ax1.semilogy(layer_range, SIGMOID_MAX_GRAD ** layer_range,
             color="#E74C3C", linewidth=2.5, label="잔차 없음 (Sigmoid)")
ax1.semilogy(layer_range, np.abs(mean_residual_by_depth),
             color="#2980B9", linewidth=2.5, label="잔차 있음")
ax1.axhline(y=1e-10, color="#E67E22", linestyle=":", linewidth=1.5,
            label="학습 불가 기준")
ax1.fill_between(layer_range, 1e-35, 1e-10, alpha=0.08, color="#E74C3C")
ax1.set_xlabel("레이어 수", fontsize=11)
ax1.set_ylabel("그래디언트 크기 (로그 스케일)", fontsize=11)
ax1.set_title("레이어 수 vs 그래디언트 크기")
ax1.legend(fontsize=9)

# 가운데: 잔차 있을 때 그래디언트 분포
ax2 = axes[1]
ax2.hist(residual_grads, bins=60, color="#3498DB", alpha=0.75,
         edgecolor="white", linewidth=0.5)
ax2.axvline(np.mean(residual_grads), color="#E74C3C", linestyle="--",
            linewidth=2.5, label=f"평균: {np.mean(residual_grads):.3f}")
ax2.axvline(np.median(residual_grads), color="#E67E22", linestyle="-.",
            linewidth=2.5, label=f"중앙값: {np.median(residual_grads):.3f}")
ax2.axvline(1.0, color="#888888", linestyle=":", linewidth=1.5, label="기준 (1.0)")
ax2.set_xlabel("그래디언트 값", fontsize=11)
ax2.set_ylabel("실험 빈도", fontsize=11)
ax2.set_title(f"잔차 있을 때 분포\n({N_LAYERS}레이어, {N_EXPERIMENTS}회)")
ax2.legend(fontsize=9)

# 오른쪽: 아키텍처 개념도
ax3 = axes[2]
ax3.axis("off")
ax3.set_xlim(0, 10)
ax3.set_ylim(0, 12)
ax3.set_title("잔차 연결 구조\n(그래디언트 고속도로)", fontsize=11)

# 입력 박스
ax3.text(5, 1.0, "x  (입력)", ha="center", va="center", fontsize=11,
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#AED6F1",
                   edgecolor="#2980B9", lw=2))

# F(x) 레이어 박스
ax3.text(3, 5.5, "F(x) 레이어\n어텐션 / FFN", ha="center", va="center", fontsize=10,
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#FDEBD0",
                   edgecolor="#E67E22", lw=2))

# 화살표: 입력 → F(x)
ax3.annotate("", xy=(3.2, 4.3), xytext=(4.5, 1.5),
             arrowprops=dict(arrowstyle="->", color="#E67E22", lw=2))

# 화살표: F(x) → 덧셈 노드
ax3.annotate("", xy=(4.65, 8.0), xytext=(3.3, 6.8),
             arrowprops=dict(arrowstyle="->", color="#E67E22", lw=2))

# 잔차 연결 화살표 (직접 경로)
ax3.annotate("", xy=(5.35, 7.7), xytext=(5.7, 1.5),
             arrowprops=dict(arrowstyle="->", color="#27AE60", lw=3,
                             connectionstyle="arc3,rad=-0.38"))

# 고속도로 레이블
ax3.text(8.5, 4.5, "그래디언트\n고속도로\n(+x)", ha="center", va="center",
         fontsize=10, color="#27AE60", fontweight="bold",
         bbox=dict(boxstyle="round", facecolor="#EAFAF1", alpha=0.9))

# 덧셈 노드
circle = plt.Circle((5, 8.3), 0.55, color="#F39C12", zorder=5)
ax3.add_patch(circle)
ax3.text(5, 8.3, "+", ha="center", va="center", fontsize=20,
         fontweight="bold", zorder=6)

# 출력
ax3.annotate("", xy=(5, 10.5), xytext=(5, 8.85),
             arrowprops=dict(arrowstyle="->", color="#2C3E50", lw=2))
ax3.text(5, 11.2, "F(x) + x  (출력)", ha="center", va="center", fontsize=11,
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#A9DFBF",
                   edgecolor="#1E8449", lw=2))

plt.tight_layout()
plt.savefig("sec2_residual_connections.png", dpi=150, bbox_inches="tight")
plt.show()
print("그래프 저장: sec2_residual_connections.png")

## 섹션 3: 소프트맥스의 그래디언트 — 야코비안 행렬

### 트랜스포머 어텐션 복습

```
Attention(Q, K, V) = softmax( QKᵀ / √d_k ) × V
                      ↑
                  이 부분의 그래디언트가 중요합니다
```

### 야코비안(Jacobian)이란?

입력도 벡터이고 출력도 벡터일 때,  
**모든 입출력 조합의 편미분을 행렬로 표현**한 것입니다.

```
소프트맥스 입력: z = [z₁, z₂, z₃, z₄]   (어텐션 점수)
소프트맥스 출력: s = [s₁, s₂, s₃, s₄]   (어텐션 가중치, 합=1)

야코비안 J[i, j] = ∂sᵢ / ∂zⱼ
                   ↑ "j번째 입력이 i번째 출력에 미치는 영향"
```

### 소프트맥스 야코비안의 공식

```
i = j 일 때 (자기 자신):   J[i,i] = sᵢ × (1 - sᵢ)
i ≠ j 일 때 (다른 원소):  J[i,j] = -sᵢ × sⱼ
```

### 왜 문제가 될까?

어텐션이 **한 토큰에 집중**될수록:
- 그 토큰의 가중치 sᵢ → 1에 수렴
- J[i,i] = sᵢ × (1 - sᵢ) → 0 (예: 0.99 × 0.01 = **0.0099**!)
- 나머지 토큰의 가중치 sⱼ → 0에 수렴
- 모든 야코비안 원소 → 0

→ 어텐션이 집중될수록 역전파 그래디언트가 작아집니다!

> **트랜스포머의 해결책**: 스케일 팩터 `1/√d_k`  
> 어텐션 점수를 작게 만들어 소프트맥스가 너무 뾰족해지지 않게 합니다.

In [ ]:
# ============================================================
# 섹션 3: 소프트맥스 야코비안 구현 및 분석
# ============================================================

print("=" * 55)
print("  섹션 3: 소프트맥스 야코비안 분석")
print("=" * 55)

# ── 소프트맥스 함수 ──────────────────────────────────────

def softmax(x):
    # 수치 안정성: 최댓값을 먼저 뺀 후 계산
    # 이유: exp(큰 수)는 오버플로우 → exp(x - max(x))로 방지
    # 수학적으로는 동일: exp(a-c)/Σexp(b-c) = exp(a)/Σexp(b)
    shifted = x - np.max(x)
    e = np.exp(shifted)
    return e / e.sum()

# ── 소프트맥스 야코비안 함수 ─────────────────────────────

def softmax_jacobian(s):
    # s: softmax 출력값 (확률 벡터)
    # 반환: n×n 야코비안 행렬
    #
    # 공식:
    #   J[i, i] = s[i] × (1 - s[i])   ← 대각 원소
    #   J[i, j] = -s[i] × s[j]          ← 비대각 원소
    #
    # 왜 이런 공식이 나올까?
    #   softmax(z_i) = exp(z_i) / Σ exp(z_k)
    #   이를 z_j로 편미분하면 위 공식이 유도됨
    n = len(s)
    J = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                # 자기 자신에 대한 미분:
                # s_i가 0.5일 때 최대(0.25), 0이나 1에 가까울수록 0
                J[i, j] = s[i] * (1.0 - s[i])
            else:
                # 다른 원소에 대한 미분:
                # 음수 → 한 원소 증가 시 다른 원소 감소 (합이 1이어야 하므로)
                J[i, j] = -s[i] * s[j]
    return J

# ── 구체적 예시: 4개 토큰 ────────────────────────────────

print()
print("  [예시: 4개 토큰의 어텐션]")
print("  " + "-" * 48)

# 어텐션 점수 (Query × Key^T / √d_k 의 결과라고 가정)
scores = np.array([2.0, 1.0, 0.5, 0.0])
print(f"  어텐션 점수 (z): {scores}")
print(f"  → 토큰0이 가장 관련 높고, 토큰3이 가장 낮음")

attn = softmax(scores)
print()
print(f"  소프트맥스 후 가중치 (s):")
for i, a in enumerate(attn):
    bar = "█" * int(a * 30)
    print(f"    토큰{i}: {a:.4f}  {bar}")
print(f"    합계: {attn.sum():.4f}  (항상 1.0)")

J = softmax_jacobian(attn)
print()
print("  야코비안 행렬 J (행=출력차원, 열=입력차원):")
header = "         " + "".join([f"  z{j}    " for j in range(4)])
print("  " + header)
for i, row in enumerate(J):
    row_str = "  ".join([f"{v:+7.4f}" for v in row])
    print(f"    s{i} │ {row_str}")

frob_norm = np.linalg.norm(J, "fro")
print()
print(f"  야코비안 프로베니우스 노름 ||J|| = {frob_norm:.4f}")
print(f"  → 이 값이 클수록 역전파 시 그래디언트가 잘 전달됨")

In [ ]:
# ── 온도(Temperature)에 따른 어텐션 변화 ───────────────

print()
print("  [온도(Temperature)에 따른 어텐션 분포와 그래디언트]")
print("  " + "-" * 55)
print("  온도 T를 조절하면 어텐션 집중도를 바꿀 수 있습니다:")
print("    낮은 T (1/T 큼): 뾰족한(sharp) 분포  → 그래디언트 ↓")
print("    높은 T (1/T 작음): 균등한(flat) 분포  → 그래디언트 ↑")
print()

TEMPERATURES = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
temp_results = []

print(f"  {'온도 T':>6} │ {'최대 가중치':>11} │ {'엔트로피 H':>10} │ {'||J||':>8} │ 상태")
print("  " + "-" * 68)

for T in TEMPERATURES:
    # 온도 적용: 점수에 1/T를 곱하는 것과 동일
    # T < 1: 점수 차이 증폭 → 특정 토큰에 집중
    # T > 1: 점수 차이 축소 → 모든 토큰에 균등 분배
    scaled_scores = scores / T
    a = softmax(scaled_scores)
    J_t = softmax_jacobian(a)

    # 엔트로피: 분포의 균일도 (-Σ pᵢ log pᵢ)
    # 완전 균등: H = log(4) ≈ 1.386 / 완전 집중: H = 0
    H = -np.sum(a * np.log(a + 1e-12))
    jnorm = np.linalg.norm(J_t, "fro")
    max_w = np.max(a)

    status_map = {True: "⚠️  집중", False: "✓  균등"}
    status = status_map[max_w > 0.8]
    if 0.4 <= max_w <= 0.8:
        status = "→  중간"

    temp_results.append({"T": T, "attn": a, "H": H, "jnorm": jnorm})
    print(f"  T={T:>4.1f}  │ {max_w:>11.4f} │ {H:>10.4f} │ {jnorm:>8.4f} │ {status}")

print()
print("  핵심 관찰:")
print("    집중된 어텐션(낮은 T) → ||J|| 감소 → 역전파 그래디언트 약해짐!")
print("    트랜스포머는 1/√d_k 스케일링으로 이를 완화합니다")

# ── 시각화 ────────────────────────────────────────────────

SHOW_TEMPS = [0.1, 1.0, 10.0]
token_names = ["토큰0", "토큰1", "토큰2", "토큰3"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("섹션 3: 온도에 따른 어텐션과 야코비안 변화", fontsize=13,
             fontweight="bold", y=1.01)

for col_i, T_show in enumerate(SHOW_TEMPS):
    res = next(r for r in temp_results if r["T"] == T_show)
    a_show = res["attn"]
    J_show = softmax_jacobian(a_show)

    ax_top = axes[0][col_i]
    ax_bot = axes[1][col_i]

    # 위 행: 어텐션 가중치 막대 그래프
    colors_bar = ["#E74C3C" if v == max(a_show) else "#AED6F1" for v in a_show]
    bars = ax_top.bar(token_names, a_show, color=colors_bar, alpha=0.85,
                      edgecolor="white", linewidth=1.5)
    ax_top.axhline(0.25, color="#888", linestyle="--", alpha=0.5, linewidth=1.5,
                   label="균등 (0.25)")
    for bar, val in zip(bars, a_show):
        ax_top.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.02,
                    f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax_top.set_ylim(0, 1.15)
    ax_top.set_title(f"온도 T = {T_show}  |  H = {res['H']:.3f}", fontsize=11)
    ax_top.set_ylabel("어텐션 가중치", fontsize=10)

    # 상태 레이블
    if T_show == 0.1:
        label_t, label_c, label_bg = "⚠️ 집중: 그래디언트 위험!", "#C0392B", "#FDEDEC"
    elif T_show == 1.0:
        label_t, label_c, label_bg = "→ 보통 상태", "#2471A3", "#D6EAF8"
    else:
        label_t, label_c, label_bg = "✓ 균등: 그래디언트 양호", "#1E8449", "#D5F5E3"
    ax_top.text(0.5, 0.97, label_t, transform=ax_top.transAxes,
                ha="center", va="top", fontsize=9, color=label_c,
                bbox=dict(boxstyle="round", facecolor=label_bg, alpha=0.9))
    if col_i == 0:
        ax_top.legend(fontsize=8, loc="upper right")

    # 아래 행: 야코비안 히트맵
    im = ax_bot.imshow(J_show, cmap="RdBu_r", vmin=-0.25, vmax=0.25, aspect="auto")
    plt.colorbar(im, ax=ax_bot, shrink=0.8)
    for ri in range(4):
        for ci in range(4):
            color_txt = "white" if abs(J_show[ri, ci]) > 0.1 else "black"
            ax_bot.text(ci, ri, f"{J_show[ri,ci]:.3f}",
                        ha="center", va="center", fontsize=8.5, color=color_txt)
    ax_bot.set_title(f"야코비안  ||J|| = {res['jnorm']:.4f}", fontsize=11)
    ax_bot.set_xlabel("입력 차원 (점수 z)", fontsize=9)
    ax_bot.set_ylabel("출력 차원 (가중치 s)", fontsize=9)
    ax_bot.set_xticks(range(4))
    ax_bot.set_yticks(range(4))
    ax_bot.set_xticklabels([f"z{i}" for i in range(4)])
    ax_bot.set_yticklabels([f"s{i}" for i in range(4)])

plt.tight_layout()
plt.savefig("sec3_softmax_jacobian.png", dpi=150, bbox_inches="tight")
plt.show()
print("그래프 저장: sec3_softmax_jacobian.png")

## 섹션 4: 그래디언트 폭발과 클리핑

### 잔차 연결의 부작용?

잔차 연결이 소실 문제는 해결하지만, 반대 상황도 생길 수 있습니다.

잔차 연결의 그래디언트: `grad × (∂F/∂x + 1)`

만약 ∂F/∂x 가 크다면:
- 레이어마다 그래디언트가 1보다 훨씬 큰 값을 곱하게 됨
- 레이어를 거칠수록 그래디언트가 폭발적으로 증가 → **그래디언트 폭발!**
- 가중치 업데이트가 너무 커져서 학습이 불안정해짐

### 그래디언트 클리핑

전체 그래디언트 벡터의 **L2 노름(크기)** 을 제한합니다:

```
1단계: global_norm = √( ||∇W₁||² + ||∇W₂||² + ... )
                      ↑ 모든 파라미터 그래디언트를 하나로 합산

2단계: clip_coef = min( 1.0, max_norm / global_norm )
                        ↑ 노름이 max_norm 이하면 1.0 (변화 없음)
                        ↑ 노름이 max_norm 초과면 비율로 축소

3단계: ∇Wᵢ ← ∇Wᵢ × clip_coef
              ↑ 모든 그래디언트를 같은 비율로 축소
```

### 핵심: **방향은 유지, 크기만 제한!**

클리핑은 그래디언트 벡터의 **크기만** 줄이고 **방향은 그대로** 보존합니다.  
방향 정보가 올바른 학습 방향을 가리키기 때문입니다.

### 실제 적용 값
- 대부분의 트랜스포머 (GPT-2, BERT, T5): `max_norm = 1.0`

In [ ]:
# ============================================================
# 섹션 4: 그래디언트 클리핑 구현 및 분석
# ============================================================

print("=" * 55)
print("  섹션 4: 그래디언트 클리핑")
print("=" * 55)

# ── 그래디언트 클리핑 함수 ──────────────────────────────

def clip_grad_norm(grads, max_norm, label=""):
    # grads: 그래디언트 목록 (각각 numpy 배열)
    # max_norm: 허용할 최대 L2 노름

    # 1단계: 전체 그래디언트의 L2 노름 계산
    # sum(||g||²) = 각 그래디언트 배열의 모든 원소를 제곱하고 합산
    sum_sq = sum(np.sum(g ** 2) for g in grads)
    total_norm = np.sqrt(sum_sq)

    # 2단계: 클리핑 계수 계산
    # total_norm이 max_norm 이하면 1.0 (변화 없음)
    # total_norm이 max_norm 초과하면 max_norm/total_norm (비율 축소)
    clip_coef = min(1.0, max_norm / (total_norm + 1e-8))

    # 3단계: 모든 그래디언트에 동일한 계수 적용 (방향 유지!)
    clipped = [g * clip_coef for g in grads]

    tag = f"[{label}] " if label else ""
    print(f"  {tag}원본 노름: {total_norm:8.4f}  →  "
          f"클리핑 후: {total_norm * clip_coef:8.4f}  (max={max_norm})")
    if clip_coef < 1.0:
        print(f"    ⚠️  클리핑 발생! 계수: {clip_coef:.6f}")
    else:
        print(f"    ✓  클리핑 불필요")
    return clipped, total_norm

# ── 시나리오 비교 ────────────────────────────────────────

print()
print("  [시나리오 비교]")
print("  " + "-" * 50)
np.random.seed(0)

# 시나리오 1: 정상 그래디언트 (학습 안정기)
normal = [np.random.randn(100, 100) * 0.01,
          np.random.randn(100) * 0.01]
_, _ = clip_grad_norm(normal, 1.0, label="정상 그래디언트")

# 시나리오 2: 폭발하는 그래디언트 (불량 배치, 초기 불안정)
exploding = [np.random.randn(100, 100) * 10.0,
             np.random.randn(100) * 10.0]
clipped_g, before_norm = clip_grad_norm(exploding, 1.0, label="폭발 그래디언트")

# 방향 보존 검증
orig_flat = np.concatenate([g.flatten() for g in exploding])
clip_flat = np.concatenate([g.flatten() for g in clipped_g])
cos_sim = (np.dot(orig_flat, clip_flat) /
           (np.linalg.norm(orig_flat) * np.linalg.norm(clip_flat) + 1e-8))
print()
print(f"  방향 보존 검증 (코사인 유사도):")
print(f"    클리핑 전후 방향 일치도: {cos_sim:.8f}")
print(f"    → 1.0에 가까울수록 방향이 완벽히 보존됨 ✓")

# ── 시각화 ───────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("섹션 4: 그래디언트 클리핑", fontsize=13, fontweight="bold", y=1.01)

# 왼쪽: 클리핑 함수
ax1 = axes[0]
x_norm = np.linspace(0, 12, 300)
for mn, col, ls in [(0.5, "#E74C3C", "-"), (1.0, "#2980B9", "-"),
                     (3.0, "#27AE60", "--"), (6.0, "#8E44AD", ":")]:
    ax1.plot(x_norm, np.minimum(x_norm, mn), color=col, linewidth=2.5,
             linestyle=ls, label=f"max_norm = {mn}")
ax1.plot(x_norm, x_norm, "k-", alpha=0.25, linewidth=1.5, label="클리핑 없음")
ax1.set_xlabel("원본 그래디언트 노름", fontsize=11)
ax1.set_ylabel("클리핑 후 노름", fontsize=11)
ax1.set_title("그래디언트 클리핑 함수\nclip(norm, max_norm)")
ax1.legend(fontsize=9)

# 가운데: 2D 방향 보존 시각화
ax2 = axes[1]
orig_2d = np.array([8.0, 6.0])           # 노름 = 10
max_n_vis = 1.0
clip_2d = orig_2d * (max_n_vis / np.linalg.norm(orig_2d))

ax2.quiver(0, 0, orig_2d[0], orig_2d[1],
           angles="xy", scale_units="xy", scale=1,
           color="#E74C3C", width=0.025, alpha=0.85,
           label=f"원본 (노름 = {np.linalg.norm(orig_2d):.0f})")
ax2.quiver(0, 0, clip_2d[0], clip_2d[1],
           angles="xy", scale_units="xy", scale=1,
           color="#2980B9", width=0.03,
           label=f"클리핑 후 (노름 = {max_n_vis})")
theta = np.linspace(0, 2 * np.pi, 200)
ax2.fill(np.cos(theta), np.sin(theta), alpha=0.1, color="#2980B9")
circle_vis = plt.Circle((0, 0), max_n_vis, fill=False, color="#2980B9",
                          linestyle="--", linewidth=2)
ax2.add_patch(circle_vis)
ax2.set_xlim(-1.5, 11)
ax2.set_ylim(-1.5, 8)
ax2.set_aspect("equal")
ax2.set_xlabel("그래디언트 차원 1", fontsize=11)
ax2.set_ylabel("그래디언트 차원 2", fontsize=11)
ax2.set_title("방향 유지, 크기만 조정\n(2D 예시)")
ax2.legend(fontsize=9)
ax2.text(6.0, 5.5, f"방향 θ 동일!\n크기만\n{np.linalg.norm(orig_2d):.0f} → {max_n_vis}",
         fontsize=10, ha="center", color="#8E44AD",
         bbox=dict(boxstyle="round", facecolor="#F9EBEA", alpha=0.9))

# 오른쪽: 학습 중 그래디언트 노름 시뮬레이션
ax3 = axes[2]
np.random.seed(77)
n_step = 300
t = np.arange(n_step)
base_curve = 3.0 * np.exp(-t / 80) + 0.4 + 0.25 * np.abs(np.random.randn(n_step))
spike_pos = [35, 95, 170, 235]
for sp in spike_pos:
    base_curve[sp] += np.random.uniform(6, 14)
max_n_plot = 1.0
clipped_curve = np.minimum(base_curve, max_n_plot)

ax3.plot(t, base_curve, color="#E74C3C", alpha=0.65, linewidth=1.2,
         label="원본 그래디언트 노름")
ax3.plot(t, clipped_curve, color="#2980B9", alpha=0.9, linewidth=2,
         label="클리핑 후")
ax3.axhline(max_n_plot, color="#E67E22", linestyle="--", linewidth=2.5,
            label=f"max_norm = {max_n_plot}")
for sp in spike_pos:
    ax3.annotate("스파이크\n→ 억제",
                 xy=(sp, max_n_plot + 0.05), xytext=(sp + 18, 2.5),
                 arrowprops=dict(arrowstyle="->", color="#C0392B", lw=1.5),
                 fontsize=8.5, color="#C0392B",
                 bbox=dict(boxstyle="round", facecolor="#FDEDEC", alpha=0.85))
ax3.set_xlabel("학습 스텝", fontsize=11)
ax3.set_ylabel("그래디언트 노름", fontsize=11)
ax3.set_title("학습 중 클리핑 효과\n스파이크가 억제됩니다")
ax3.legend(fontsize=9)
ax3.set_ylim(0, max(base_curve) * 1.05)

plt.tight_layout()
plt.savefig("sec4_gradient_clipping.png", dpi=150, bbox_inches="tight")
plt.show()
print("그래프 저장: sec4_gradient_clipping.png")

## 섹션 5: 학습률 워밍업 (Learning Rate Warmup)

### 학습 초기의 상황

딥러닝 학습을 처음 시작할 때:
1. **가중치 무작위 초기화** → 모델이 아직 아무것도 모르는 상태
2. **레이어 정규화 통계 불확실** → 배치마다 평균·분산이 크게 달라짐
3. **그래디언트 방향 불안정** → 역전파 신호가 좋은 방향을 가리키지 않음

→ 이 상태에서 **큰 학습률** 을 쓰면?  
→ 가중치가 엉뚱한 방향으로 **크게** 업데이트됨 → 발산 위험!

### 해결책: 작은 학습률로 시작, 점진적 증가

```
                 ★ 최대 학습률 (warmup 종료)
                /\
               /  \
              /    \___
             /         \___
            /               \____
           /                      \____
──────────/──────────│──────────────────────→ step
          워밍업      warmup_steps          감소 구간
```

### 트랜스포머 원논문의 공식 (Vaswani et al., 2017)

```
lr(step) = d_model^{-0.5} × min( step^{-0.5},  step × warmup^{-1.5} )
                                  ↑역제곱근감소   ↑선형증가(워밍업)
```

- `step < warmup`: 선형 증가 항이 더 작음 → **학습률 선형 증가**
- `step = warmup`: 두 항이 같아지는 지점 → **최대 학습률 도달**
- `step > warmup`: 역제곱근 항이 더 작음 → **학습률 점진적 감소**

In [ ]:
# ============================================================
# 섹션 5: 학습률 워밍업 구현 및 시각화
# ============================================================

print("=" * 55)
print("  섹션 5: 학습률 워밍업")
print("=" * 55)

# ── 트랜스포머 학습률 스케줄 함수 ───────────────────────

def transformer_lr(step, d_model=512, warmup=4000):
    # Vaswani et al. (2017) Attention Is All You Need
    #
    # 수식:
    #   lr = d_model^{-0.5} × min(step^{-0.5}, step × warmup^{-1.5})
    #
    # 두 항:
    #   term1 = step^{-0.5}         → step이 커질수록 감소 (역제곱근)
    #   term2 = step × warmup^{-1.5} → step이 커질수록 증가 (선형)
    #
    # min()으로 두 항 중 작은 쪽 선택:
    #   step < warmup → term2가 더 작음 → 선형 증가 지배
    #   step > warmup → term1이 더 작음 → 역제곱근 감소 지배
    if step <= 0:
        return 0.0
    term1 = step ** (-0.5)
    term2 = step * (warmup ** (-1.5))
    return (d_model ** (-0.5)) * min(term1, term2)

# ── 단계별 학습률 출력 ──────────────────────────────────

print()
print("  [단계별 학습률 (d_model=512, warmup=4000)]")
print("  " + "-" * 60)

warmup_steps = 4000
max_lr_val = transformer_lr(warmup_steps)

milestone_steps = [1, 100, 500, 1000, 2000, 4000, 6000, 10000, 20000, 40000]
print(f"  {'스텝':>8} │ {'학습률':>12} │ {'최대 대비':>8} │ 단계")
print("  " + "-" * 58)

for step in milestone_steps:
    lr = transformer_lr(step)
    ratio = lr / max_lr_val

    if step < warmup_steps:
        phase = f"워밍업 ({step / warmup_steps:.0%})"
    elif step == warmup_steps:
        phase = "★ 최대 학습률"
    else:
        phase = f"감소 단계"

    bar = "▓" * max(1, int(ratio * 18)) + "░" * (18 - max(1, int(ratio * 18)))
    print(f"  step {step:>6} │ {lr:.7f} │ {bar} │ {phase}")

print(f"  최대 학습률: {max_lr_val:.7f}  (step={warmup_steps})")

# ── 다양한 설정 비교 ────────────────────────────────────

print()
print("  [설정별 학습률 비교]")
lr_configs = [
    {"d_model": 512,  "warmup": 4000, "label": "Base  (d512,  warmup=4k)"},
    {"d_model": 512,  "warmup": 1000, "label": "빠른  (d512,  warmup=1k)"},
    {"d_model": 1024, "warmup": 4000, "label": "큰모델(d1024, warmup=4k)"},
]
cmp_steps = [500, 2000, 4000, 10000, 40000]
print(f"  {'설정':<30}", end="")
for s in cmp_steps:
    print(f"  step={s:>6}", end="")
print()
print("  " + "-" * 90)
for cfg in lr_configs:
    print(f"  {cfg['label']:<30}", end="")
    for s in cmp_steps:
        lr_v = transformer_lr(s, d_model=cfg["d_model"], warmup=cfg["warmup"])
        print(f"  {lr_v:.5f}  ", end="")
    print()

# ── 시각화 ───────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("섹션 5: 학습률 워밍업", fontsize=13, fontweight="bold", y=1.01)

# 왼쪽: 학습률 스케줄 곡선
ax1 = axes[0]
steps_vis = np.arange(1, 60001)
cfg_styles = [
    {"d_model": 512,  "warmup": 4000, "label": "Base (d512, warmup=4k)",
     "color": "#2980B9", "lw": 2.5, "ls": "-"},
    {"d_model": 512,  "warmup": 1000, "label": "빠른 워밍업 (warmup=1k)",
     "color": "#27AE60", "lw": 2, "ls": "--"},
    {"d_model": 1024, "warmup": 4000, "label": "큰 모델 (d1024)",
     "color": "#E74C3C", "lw": 2, "ls": "-."},
]
for cfg in cfg_styles:
    lr_curve = np.array([transformer_lr(s, d_model=cfg["d_model"], warmup=cfg["warmup"])
                         for s in steps_vis])
    ax1.plot(steps_vis, lr_curve, color=cfg["color"], linewidth=cfg["lw"],
             linestyle=cfg["ls"], label=cfg["label"], alpha=0.88)

ax1.axvspan(0, 4000, alpha=0.07, color="#F39C12")
ax1.axvline(4000, color="#E67E22", linestyle=":", linewidth=2, alpha=0.7)
ax1.axvline(1000, color="#27AE60", linestyle=":", linewidth=1.5, alpha=0.5)
ax1.text(2000, 5.5e-5, "워밍업\n(선형 증가)", ha="center", fontsize=10,
         color="#E67E22", fontweight="bold",
         bbox=dict(boxstyle="round", facecolor="#FEF9E7", alpha=0.9))
ax1.text(35000, 1.2e-5, "감소\n(역제곱근)", ha="center", fontsize=10,
         color="#2471A3", fontweight="bold",
         bbox=dict(boxstyle="round", facecolor="#EBF5FB", alpha=0.9))
ax1.set_xlabel("학습 스텝", fontsize=11)
ax1.set_ylabel("학습률", fontsize=11)
ax1.set_title("트랜스포머 학습률 스케줄\n워밍업 → 최대 → 역제곱근 감소")
ax1.legend(fontsize=9)

# 오른쪽: 워밍업 유무에 따른 학습 안정성
ax2 = axes[1]
np.random.seed(SEED)
n_sim = 400

# 워밍업 없는 경우: 초기부터 큰 학습률 → 불안정
no_wu = [2.0]
for step in range(1, n_sim):
    instability = np.exp(-step / 40)
    spike = 0.9 * instability if step < 25 else 0.0
    noise = np.random.randn() * (0.25 * instability + 0.03)
    no_wu.append(max(0.1, no_wu[-1] - 0.007 + noise + spike))

# 워밍업 있는 경우: 안정적 시작 후 감소
with_wu = [2.0]
for step in range(1, n_sim):
    lr_wu = transformer_lr(step, warmup=80) * 40000
    noise = np.random.randn() * 0.035
    with_wu.append(max(0.1, with_wu[-1] - 0.012 * lr_wu + noise))

ax2.plot(no_wu, color="#E74C3C", alpha=0.8, linewidth=2, label="워밍업 없음 (고정 lr)")
ax2.plot(with_wu, color="#2980B9", alpha=0.85, linewidth=2, label="워밍업 있음")
ax2.axvspan(0, 80, alpha=0.09, color="#27AE60")
ax2.text(40, 2.35, "워밍업\n구간", ha="center", fontsize=9.5, color="#1E8449",
         bbox=dict(boxstyle="round", facecolor="#D5F5E3", alpha=0.85))
ax2.annotate("초기\n불안정",
             xy=(15, no_wu[15]), xytext=(60, no_wu[15] + 0.5),
             arrowprops=dict(arrowstyle="->", color="#C0392B", lw=1.5),
             fontsize=9, color="#C0392B",
             bbox=dict(boxstyle="round", facecolor="#FDEDEC", alpha=0.85))
ax2.set_xlabel("학습 스텝", fontsize=11)
ax2.set_ylabel("학습 손실 (Loss)", fontsize=11)
ax2.set_title("워밍업 유무에 따른\n학습 안정성 비교 (시뮬레이션)")
ax2.legend(fontsize=10)
ax2.set_ylim(0.05, 2.9)

plt.tight_layout()
plt.savefig("sec5_lr_warmup.png", dpi=150, bbox_inches="tight")
plt.show()
print("그래프 저장: sec5_lr_warmup.png")

In [ ]:
# ============================================================
# 종합 실험: 모든 기법을 조합했을 때 vs 없을 때
#
# 지금까지 배운 4가지 기법의 효과를 한 눈에 비교합니다.
# - 잔차 연결: 그래디언트 소실 방지
# - 그래디언트 클리핑: 폭발 방지
# - 학습률 워밍업: 초기 안정화
# (참고로 소프트맥스 스케일링은 모든 경우에 동일하게 적용)
# ============================================================

print("=" * 55)
print("  종합 실험: 기법 조합 비교")
print("=" * 55)

def simulate_training(n_steps=400, n_layers=12,
                       use_residual=True, use_clipping=True, use_warmup=True,
                       seed=42):
    # 간소화된 트랜스포머 학습 시뮬레이션
    # 각 기법의 효과를 직관적으로 보여주기 위한 단순 모델
    np.random.seed(seed)
    losses = [2.0]
    grad_norms = []

    for step in range(1, n_steps + 1):
        # 학습률 결정
        if use_warmup:
            lr = transformer_lr(step, warmup=80) * 30000
        else:
            lr = 1.0  # 고정 (상대적 스케일)

        # 기본 그래디언트 크기 (손실에 비례)
        base_g = 1.5 / (losses[-1] + 0.1)
        base_g *= (1.0 + 0.4 * np.abs(np.random.randn()))

        # 잔차 연결 효과: 없으면 그래디언트 소실
        if use_residual:
            effective_g = base_g * np.exp(0.04 * np.random.randn())
        else:
            # n_layers 레이어 통과 후 소실 효과 반영
            vanish = SIGMOID_MAX_GRAD ** (n_layers // 3)
            effective_g = base_g * vanish * (1.0 + np.abs(np.random.randn()) * 0.1)

        # 가끔 스파이크 (불량 배치)
        if np.random.rand() < 0.05:
            effective_g *= np.random.uniform(5, 18)

        # 클리핑 적용
        actual_g = min(effective_g, 1.0) if use_clipping else effective_g
        grad_norms.append(actual_g)

        # 손실 업데이트 (단순화)
        update_mag = lr * actual_g
        stability = 1.0 / (1.0 + max(0.0, update_mag * 0.8 - 0.15))
        noise = np.random.randn() * (0.04 / max(stability, 0.1))
        loss_delta = 0.009 * stability
        losses.append(max(0.08, losses[-1] - loss_delta + noise))

    return np.array(losses), np.array(grad_norms)

# ── 5가지 설정 비교 ─────────────────────────────────────

exp_configs = [
    {"label": "모든 기법 사용 (이상적 트랜스포머)",
     "res": True,  "clip": True,  "wu": True,  "color": "#2980B9"},
    {"label": "잔차 없음         (그래디언트 소실)",
     "res": False, "clip": True,  "wu": True,  "color": "#E74C3C"},
    {"label": "클리핑 없음       (폭발 가능성)",
     "res": True,  "clip": False, "wu": True,  "color": "#E67E22"},
    {"label": "워밍업 없음       (초기 불안정)",
     "res": True,  "clip": True,  "wu": False, "color": "#27AE60"},
    {"label": "모두 없음         (기본 딥러닝)",
     "res": False, "clip": False, "wu": False, "color": "#8E44AD"},
]

print()
print(f"  {'설정':<45} │ 최종 손실")
print("  " + "-" * 58)

all_losses = {}
all_grads = {}

for cfg in exp_configs:
    losses_r, grads_r = simulate_training(
        use_residual=cfg["res"], use_clipping=cfg["clip"], use_warmup=cfg["wu"])
    all_losses[cfg["label"]] = losses_r
    all_grads[cfg["label"]] = grads_r
    print(f"  {cfg['label']:<45} │ {losses_r[-1]:.4f}")

# ── 시각화 ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("종합 실험: 그래디언트 안정화 기법 비교 (시뮬레이션)",
             fontsize=13, fontweight="bold", y=1.01)

ax1, ax2 = axes

for cfg in exp_configs:
    lbl = cfg["label"]
    ax1.plot(all_losses[lbl], color=cfg["color"], linewidth=2,
             label=lbl, alpha=0.85)
    ax2.plot(all_grads[lbl], color=cfg["color"], linewidth=1.5,
             label=lbl, alpha=0.75)

ax1.set_xlabel("학습 스텝", fontsize=11)
ax1.set_ylabel("학습 손실", fontsize=11)
ax1.set_title("학습 곡선 비교")
ax1.legend(fontsize=8, loc="upper right")
ax1.set_ylim(0, 2.8)

ax2.axhline(1.0, color="black", linestyle="--", linewidth=2,
            alpha=0.7, label="클리핑 임계값 (1.0)")
ax2.set_xlabel("학습 스텝", fontsize=11)
ax2.set_ylabel("그래디언트 노름", fontsize=11)
ax2.set_title("그래디언트 노름 비교")
ax2.legend(fontsize=8, loc="upper right")
ax2.set_ylim(0, min(8.0, np.percentile(
    np.concatenate(list(all_grads.values())), 99) * 1.1))

plt.tight_layout()
plt.savefig("sec_comprehensive.png", dpi=150, bbox_inches="tight")
plt.show()

print()
print("=" * 55)
print("  종합 결론:")
print("  ✓ 잔차 연결  : 그래디언트 소실 방지 (가장 핵심!)")
print("  ✓ 그래디언트 클리핑: 스파이크/폭발 억제")
print("  ✓ 학습률 워밍업: 초기 발산 방지")
print("  → 세 기법이 함께 작동할 때 가장 안정적인 학습!")
print("=" * 55)

## 종합 요약

### 이 노트북에서 배운 것들

| 문제 | 원인 | 해결책 | 핵심 수식 |
|------|------|--------|-----------|
| 그래디언트 소실 | 미분값의 연속 곱 → 0 | **잔차 연결** | `∂out/∂in = ∂F/∂in + 1` |
| 어텐션 그래디언트 | 집중 어텐션 → 야코비안↓ | **스케일 팩터 1/√d_k** | `J[i,j] = sᵢδᵢⱼ - sᵢsⱼ` |
| 그래디언트 폭발 | 잔차 연결 + 큰 미분값 | **그래디언트 클리핑** | `coef = min(1, max_norm/\|\|g\|\|)` |
| 초기 학습 불안정 | 무작위 초기화 | **학습률 워밍업** | `lr = d^{-0.5}·min(t^{-0.5}, t·w^{-1.5})` |

### 실제 트랜스포머 레이어 코드

```python
class TransformerLayer(nn.Module):
    def forward(self, x):
        # 잔차 연결: x = x + ...
        x = x + self.attn(self.norm1(x))   # 어텐션 레이어 + 잔차
        x = x + self.ffn(self.norm2(x))    # FFN 레이어 + 잔차
        return x

# 학습 루프
for step, batch in enumerate(dataloader):
    loss = model(batch)
    loss.backward()

    # 그래디언트 클리핑
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()          # AdamW / Adam
    scheduler.step()          # 학습률 워밍업 + 감소
    optimizer.zero_grad()
```

### 다음 단계

이 노트북의 내용을 이해했다면, 아래 주제를 학습해보세요:

1. **레이어 정규화(LayerNorm)**: 내부 공변량 이동 문제 해결
2. **AdamW 옵티마이저**: 적응적 학습률 + 가중치 감쇠 (weight decay)
3. **혼합 정밀도 학습(FP16)**: 속도 향상 시 그래디언트 스케일링 필요
4. **Flash Attention**: 어텐션 연산 최적화와 수치 안정성

---
*Tutorial: `adv-3-1` | Section: `adv-3-1-1` | 잔차 연결과 그래디언트 고속도로*